# 17 · External sources, ZIP sampling, CAT zone, bad values, imputation

Starting point: `SPEC7` (34 parameters, 15-fold 0.7243). Questions:
1. Which **density source** did the generator use? Ours is ACS 2018 population ÷ 2018 land area.
2. How were **ZIPs and quotes sampled**?
3. Is **CAT zone** geographic?
4. Does our **capping** of bad values (driver age > 100, safety < 0) match the generator? Are there other planted values?
5. Can **missing values** be imputed better?

Helpers:
- `external.py`: downloads, cached in `data/external/`;
- `geo_tests.py`: group residual tests, density joins;
- `impute.py`: imputation variants; `impute.core` feeds `formula.prep`.

In [1]:
import sys, json, glob, zipfile
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, log_loss
sys.path.insert(0, '.')
import common as C, features as FE, formula as FM, formula_steps as FS, external as E, geo_tests as G, impute as I
from formula_steps import t
L, y, lab, hold = FS.L, FS.y, FS.lab, FS.hold
era = L.ERA.astype(str).to_numpy()
zkey = lambda D: D.ZIP_CD.astype(str).where(D.ZIP_CD.astype(str).str.fullmatch(r'\d{5}')).to_numpy()
z = zkey(L); z3 = np.array([s[:3] for s in L.ZIP_CD.astype(str)])
LD = G.add_density(L)
cnty = LD.COUNTY_FIPS.fillna('NA').to_numpy()
g10, g18, g20 = E.gazetteer(2010), E.gazetteer(2018), E.gazetteer(2020)
POP = {f'acs{yr}': E.acs_pop(yr).POP for yr in range(2011, 2021)} | {'dec2010': g10.POP10.astype(float), 'dec2020': E.dec2020_pop().POP}
AREA = {'land10': g10.ALAND_SQMI, 'land18': g18.ALAND_SQMI, 'land20': g20.ALAND_SQMI, 'tot18': g18.ALAND_SQMI + g18.AWATER_SQMI}
print({k: len(v) for k, v in POP.items()})

{'acs2011': 6690, 'acs2012': 6690, 'acs2013': 6690, 'acs2014': 6690, 'acs2015': 6690, 'acs2016': 6690, 'acs2017': 6690, 'acs2018': 6690, 'acs2019': 6690, 'acs2020': 33120, 'dec2010': 33120, 'dec2020': 33774}


## 1. Is any area-level signal left? (SPEC7 residuals vs 100 simulations)
Moment test for a random intercept per group: S = sum of residuals in the group, V = sum p(1−p), stat = Σ(S² − V) / √(2ΣV²).
- Null distribution: 100 outcome sets simulated from SPEC7, each refit.
- `sd of group effect` is the excess between-group standard deviation on the log-odds scale.

In [2]:
def group_table(spec, n_sim=100, seed=5):
    def stats(yy):
        _, p = G.fit_predict(spec, L.assign(CONVERT_IND=yy), yy)
        return {'ZIP': G.zip_effect(yy, p, z.astype(str), era), 'ZIP3': G.zip_effect(yy, p, z3, era), 'county': G.zip_effect(yy, p, cnty, era)}
    real = stats(y); _, pst = G.fit_predict(spec, L, y)
    rng = np.random.default_rng(seed); sims = [stats((rng.random(len(y)) < pst).astype(float)) for _ in range(n_sim)]
    rows = []
    for lvl in real:
        for e in real[lvl]:
            s = np.array([x[lvl][e]['stat'] for x in sims]); tt = np.array([x[lvl][e]['tau2'] for x in sims])
            rows.append({'level': lvl, 'era': e, 'real stat': real[lvl][e]['stat'], 'sim mean': s.mean(), 'sim sd': s.std(),
                         'p (sims ≥ real)': (s >= real[lvl][e]['stat']).mean(), 'sd of group effect': np.sqrt(max(real[lvl][e]['tau2'] - tt.mean(), 0))})
    return pd.DataFrame(rows)
group_table(FS.SPEC7).style.format(precision=3).hide(axis='index')

level,era,real stat,sim mean,sim sd,p (sims ≥ real),sd of group effect
ZIP,all,-1.199,-0.177,0.878,0.850,0.000
ZIP,2015,-1.559,-0.050,0.984,0.940,0.000
ZIP,2016,-0.977,0.100,0.907,0.890,0.000
ZIP,2017-18,-0.588,-0.191,0.930,0.630,0.000
ZIP3,all,-0.887,-0.234,0.857,0.720,0.000
ZIP3,2015,-0.572,-0.197,0.938,0.630,0.000
ZIP3,2016,-0.763,-0.281,0.863,0.700,0.000
ZIP3,2017-18,3.071,-0.278,0.809,0.000,0.190
county,all,-1.235,-0.212,0.912,0.880,0.000
county,2015,-0.273,-0.197,0.945,0.480,0.000


**Reading:** there's no excess at the single-ZIP level. In **2017-18** there is a regional effect at ZIP3 and county level (p ≈ 0.00 / 0.01), worth about ±0.19 log-odds between areas. Something geographic, smoother than one ZIP, matters in the late era beyond our density column.

## 2. Density sources: likelihood
Each population vintage × area source replaces `ACS_POP_DENSITY` in SPEC7 (same form, same parameter count). Table: change in full-data deviance (negative = better).

In [3]:
base_dev = G.deviance(FS.SPEC7, L, y)[0]
R = pd.DataFrame(index=list(POP), columns=list(AREA), dtype=float)
LX = L.copy()
for pk, pv in POP.items():
    for ak, av in AREA.items():
        LX['DX'] = (pv.reindex(z) / av.reindex(z)).to_numpy()
        R.loc[pk, ak] = G.deviance(G.swap(FS.SPEC7, 'ACS_POP_DENSITY', 'DX'), LX, y)[0] - base_dev
R.style.format('{:+.1f}').background_gradient(cmap='RdYlGn_r', axis=None)

,land10,land18,land20,tot18
acs2011,-27.3,-30.4,-30.0,-28.5
acs2012,-26.6,-30.3,-30.0,-29.0
acs2013,-24.3,-28.5,-28.2,-28.2
acs2014,-18.3,-24.8,-24.5,-26.2
acs2015,-16.6,-23.6,-23.3,-25.4
acs2016,-10.3,-18.4,-18.1,-21.2
acs2017,+1.4,-8.3,-8.0,-13.2
acs2018,+12.2,+0.0,+0.3,-7.1
acs2019,+13.3,+1.2,+1.5,-6.3
acs2020,-14.8,-24.4,-24.0,-27.1


In [4]:
# power check: if ACS 2018 (or ACS 2011) were the generator's input, what deviance difference would ACS 2011 vs 2018 show?
LX['P2018'] = (POP['acs2018'].reindex(z) / g18.ALAND_SQMI.reindex(z)).to_numpy(); LX['P2011'] = (POP['acs2011'].reindex(z) / g18.ALAND_SQMI.reindex(z)).to_numpy()
S18, S11 = G.swap(FS.SPEC7, 'ACS_POP_DENSITY', 'P2018'), G.swap(FS.SPEC7, 'ACS_POP_DENSITY', 'P2011')
rng = np.random.default_rng(3); out = {'real data': {'mean': G.deviance(S11, LX, y)[0] - G.deviance(S18, LX, y)[0]}}
for truth, name in ((S18, 'simulated, 2018 true'), (S11, 'simulated, 2011 true')):
    _, p = G.fit_predict(truth, LX, y); d = []
    for i in range(100):
        ys = (rng.random(len(y)) < p).astype(float); Ls = LX.assign(CONVERT_IND=ys)
        d.append(G.deviance(S11, Ls, ys)[0] - G.deviance(S18, Ls, ys)[0])
    out[name] = {'mean': np.mean(d), 'sd': np.std(d), 'min': np.min(d), 'max': np.max(d)}
pd.DataFrame(out).T.round(1)

,mean,sd,min,max
real data,-30.4,NaN,NaN,NaN
"simulated, 2018 true",32.2,9.4,2.1,52.2
"simulated, 2011 true",-31.4,9.9,-57.6,-7.1


**Reading:** older population counts fit much better: ACS 2011–2013 and the 2010 census beat ACS 2018 by 24–30 deviance points at equal complexity.
- If ACS 2018 were the generator's input, ACS 2011 would be *worse* by +32 ± 9 and never better (100 simulations).
- The real −30 matches the simulations where 2011-era data is the truth.
- The land-area source matters little (2018/2020 land is slightly better than 2010).

## 3. ZIP sampling: the selection rule identifies the exact source
- Only 1,168 of the ~6,600 ZCTAs in the 8 states appear.
- For each density variant: the best single threshold separating selected from unselected ZCTAs, and how many it misclassifies.

In [5]:
FIPS = {'01': 'AL', '09': 'CT', '12': 'FL', '13': 'GA', '27': 'MN', '34': 'NJ', '36': 'NY', '55': 'WI'}
rel = E.zcta_county_2010()
idx = pd.Index(rel[rel.STATE.isin(FIPS)].ZCTA.unique())
pol = pd.read_parquet('../data/processed/policies.parquet'); sel = idx.isin(pol.ZIP_CD.dropna().unique())
res = {}
for pk, pv in POP.items():
    for ak, av in AREA.items():
        d = (pv.reindex(idx) / av.reindex(idx)).to_numpy(); ok = np.isfinite(d); o = np.argsort(d[ok]); ds, ys = d[ok][o], sel[ok][o]
        err = np.cumsum(ys)[:-1] + (~ys[::-1]).cumsum()[::-1][1:]; k = err.argmin()
        res[(pk, ak)] = {'misclassified': int(err[k]), 'threshold (per sq mi)': (ds[k] + ds[k + 1]) / 2, 'threshold (per km²)': (ds[k] + ds[k + 1]) / 2 / 2.589988}
pd.DataFrame(res).T.sort_values('misclassified').head(12).style.format({'threshold (per sq mi)': '{:.0f}', 'threshold (per km²)': '{:.1f}'})

In [6]:
D = pd.DataFrame({'pop2010': g10.POP10.reindex(idx), 'land2018_km2': (g18.ALAND / 1e6).reindex(idx)}); D['density_km2'] = D.pop2010 / D.land2018_km2; D['selected'] = sel
print('lowest selected densities:', D[D.selected].density_km2.nsmallest(3).round(2).tolist(), '| highest unselected below 1000:', D[~D.selected & (D.density_km2 < 1000)].density_km2.nlargest(3).round(2).tolist())
print('the only exception (unselected, >= 1000/km²):'); display(D[(D.density_km2 >= 1000) & ~D.selected])
q = pol[pol.ZIP_CD.notna()]; n = q.ZIP_CD.value_counts()
S = pd.DataFrame({'n': n, 'state': q.groupby('ZIP_CD').STATE_CD.first()})
print(f'quotes per ZIP: mean {S.n.mean():.1f}, variance {S.n.var():.1f}'); display(S.groupby('state').n.agg(['size', 'mean', 'var']).round(1).T)
q = q.assign(IDN=q.POLICY_ID.str.extract(r'(\d+)')[0].astype(int)).sort_values('IDN')
print('share of consecutive IDs (within state) that change ZIP vs random order:',
      {s: (round((g.ZIP_CD.values[1:] != g.ZIP_CD.values[:-1]).mean(), 3), round(1 - ((g.ZIP_CD.value_counts() / len(g)) ** 2).sum(), 3)) for s, g in q.groupby('STATE_CD')})

lowest selected densities: [1000.76, 1001.45, 1003.26] | highest unselected below 1000: [999.22, 997.83, 997.5]
the only exception (unselected, >= 1000/km²):


,pop2010,land2018_km2,density_km2,selected
54010,50,0.035434,1411.07411,False


quotes per ZIP: mean 31.3, variance 49.6


state,AL,CT,FL,GA,MN,NJ,NY,WI
size,20,57,278,56,66,243,392,56
mean,38.5,31.8,31.9,38.2,30.2,29.7,29.3,39.6
var,32.1,23.8,33.7,52.6,35.9,29.1,56.6,40.7


share of consecutive IDs (within state) that change ZIP vs random order: {'AL': (0.956, 0.949), 'CT': (0.986, 0.982), 'FL': (0.996, 0.996), 'GA': (0.983, 0.982), 'MN': (0.992, 0.984), 'NJ': (0.995, 0.996), 'NY': (0.998, 0.997), 'WI': (0.984, 0.982)}


**Reading:**
- **The generator's density is 2010 Census population ÷ 2018 Gazetteer land area, and ZIPs were selected at exactly ≥ 1,000 people per km².** That rule misclassifies 1 of ~6,600 ZCTAs (ACS 2018: 120). The lowest selected ZIP is 1,000.8/km², the highest unselected 999.2. The one exception is a 50-person ZCTA (WI 54010), so there was probably also a minimum population.
- **Allocation:** quotes per ZIP are ~30 in the larger states and ~38–40 in AL/GA/WI. That's roughly even with mild overdispersion; the −0.44 correlation with density comes from the state mix.
- **IDs:** policy IDs are in random order within state, so they can't recover missing ZIPs.

## 4. Which columns depend on ZIP? CAT zone geography
Within-state ZIP η² of each column against a within-state permutation null (30 permutations).

In [7]:
A = pd.concat([lab, hold]); A = A[A.ZIP_CD.astype(str).str.fullmatch(r'\d{5}')].copy()
cols = ['CAT_ZONE', 'CREDIT_SCORE', 'QUOTED_AMT', 'NUM_DRIVERS', 'VEH_N_ROWS', 'DISCOUNT_IND', 'DRV_AGE_MEAN', 'DRV_HIGH_EDU_SHARE', 'DRV_ACC_FREE_MIN', 'VEH_AGE_MIN', 'QUOTE_YEAR', 'PRIOR_CLAIMS_CNT']
for c in ['PRIOR_CARRIER_GRP', 'COV_PKG_TYPE', 'QUOTE_CHANNEL']:
    for lv in A[c].astype(str).value_counts().index[:2]: A[f'{c}={lv}'] = (A[c].astype(str) == lv).astype(float); cols.append(f'{c}={lv}')
rng = np.random.default_rng(0); st = A.STATE_CD.astype(str).to_numpy(); zp = A.ZIP_CD.astype(str).to_numpy()
def eta_within(x, zz):
    s = pd.DataFrame({'x': x, 's': st, 'z': zz}).dropna(); xs = s.x - s.groupby('s').x.transform('mean')
    return 1 - ((xs - xs.groupby(s.z).transform('mean')) ** 2).sum() / (xs ** 2).sum()
rows = {}
for c in cols:
    x = pd.to_numeric(A[c], errors='coerce').to_numpy(float); e = eta_within(x, zp); null = []
    for _ in range(30):
        zz = zp.copy()
        for s_ in np.unique(st): i = np.where(st == s_)[0]; zz[i] = rng.permutation(zz[i])
        null.append(eta_within(x, zz))
    rows[c] = {'η² ZIP | state': e, 'null mean': np.mean(null), 'z': (e - np.mean(null)) / np.std(null)}
pd.DataFrame(rows).T.sort_values('z', ascending=False).style.format(precision=4)

,η² ZIP | state,null mean,z
QUOTE_YEAR,0.0367,0.0321,4.5765
QUOTE_CHANNEL=online,0.0350,0.0311,2.9093
CREDIT_SCORE,0.0347,0.0320,1.7681
COV_PKG_TYPE=Medium,0.0340,0.0319,1.6960
COV_PKG_TYPE=High,0.0338,0.0318,1.3982
PRIOR_CARRIER_GRP=Carrier_1,0.0340,0.0324,1.1976
VEH_N_ROWS,0.0326,0.0315,1.0576
DRV_ACC_FREE_MIN,0.0328,0.0314,1.0019
QUOTE_CHANNEL=agent,0.0322,0.0315,0.6493
VEH_AGE_MIN,0.0321,0.0322,-0.0778


In [8]:
A['CZ'] = pd.to_numeric(A.CAT_ZONE, errors='coerce')
display((pd.crosstab(A.STATE_CD, A.CZ, normalize='index') * 100).round(1).assign(mean=A.groupby('STATE_CD').CZ.mean().round(2)))
f = E.fema_declarations(); AD = G.add_density(A)
for c, s in [('Hurricane', 'hurricane declarations'), ('Severe Storm', 'severe-storm declarations'), ('Flood', 'flood declarations')]:
    AD[s] = AD.COUNTY_FIPS.map(f[c]).fillna(0)
r = {}
for c in ['hurricane declarations', 'severe-storm declarations', 'flood declarations', 'LAT', 'LON', 'WATER_SHARE']:
    x = AD[[c, 'CZ', 'STATE_CD']].dropna(); xc = x[c] - x.groupby('STATE_CD')[c].transform('mean'); yc = x.CZ - x.groupby('STATE_CD').CZ.transform('mean')
    r[c] = {'corr pooled': np.corrcoef(x[c], x.CZ)[0, 1], 'corr within state': np.corrcoef(xc, yc)[0, 1]}
pd.DataFrame(r).T.round(3)

CZ,1,2,3,4,5,mean
STATE_CD,,,,,,
AL,15.8,15.5,13.7,21.3,33.8,3.42
CT,24.6,26.8,19.0,14.0,15.6,2.69
FL,24.5,27.2,18.2,14.7,15.3,2.69
GA,15.5,16.4,15.5,21.2,31.4,3.36
MN,26.4,27.1,18.2,14.0,14.3,2.63
NJ,27.1,26.1,18.5,14.1,14.3,2.62
NY,20.2,22.5,17.0,18.0,22.4,3.0
WI,14.4,16.2,14.9,21.4,33.1,3.43


,corr pooled,corr within state
hurricane declarations,-0.067,0.005
severe-storm declarations,0.005,0.002
flood declarations,-0.024,0.003
LAT,0.038,0.002
LON,-0.045,-0.006
WATER_SHARE,-0.051,-0.003


**Reading:**
- **No column varies by ZIP beyond chance once state is removed.** The one mild exception is quote year (z ≈ 4).
- **CAT zone is drawn from a fixed distribution per state, of three types:**
  - AL/GA/WI skew high (mean 3.4);
  - CT/FL/MN/NJ skew low (2.65);
  - NY is uniform (3.0, the same in every NY ZIP3).
- It's unrelated to real hazards: Florida is low, and hurricane, storm and flood history have zero within-state correlation. So it's synthetic, not real hazard data.

## 5. Bad values
- Raw driver table audit; treatments of driver ages > 100.
- Log-likelihood of the affected labeled policies under SPEC7, with coefficients fixed.

In [9]:
drv = pd.concat([pd.read_csv(zipfile.ZipFile(f'../data/raw/2026-travelers-umc-{s}.zip').open(n)) for s in ['training', 'validation', 'holdout']
                 for n in zipfile.ZipFile(f'../data/raw/2026-travelers-umc-{s}.zip').namelist() if 'driver' in n and n.endswith('.csv')])
print(f'driver age > 100: {(drv.DRIVER_AGE > 100).sum()} rows on {drv[drv.DRIVER_AGE > 100].POLICY_ID.nunique()} policies (max {drv.DRIVER_AGE.max()}); '
      f'safety < 0: {(drv.SAFETY_RATING < 0).sum()} rows (min {drv.SAFETY_RATING.min()}); accident-free years > age − 16: {(drv.ACCIDENT_FREE_YRS > drv.DRIVER_AGE - 16).mean():.0%} of drivers')
q = L.QUOTED_AMT; print('quoted price quantiles 50/99/max:', np.nanquantile(q, [.5, .99, 1]).round(0))
bad = drv[drv.DRIVER_AGE > 100].POLICY_ID.unique(); i = L.POLICY_ID.isin(bad).to_numpy()
m7 = FM.Formula(FS.SPEC7, 0.1, '2era').fit(L, y)
V = {'cap at 100 (current)': lambda s: s.clip(upper=100), 'raw': lambda s: s, 'drop (missing)': lambda s: s.where(s <= 100), 'age − 100': lambda s: s.where(s <= 100, s - 100)}
rows = {}
for k, fn in V.items():
    ma = drv.assign(A=fn(drv.DRIVER_AGE)).groupby('POLICY_ID').A.mean().reindex(L.POLICY_ID).to_numpy()
    Lv = L.assign(DRV_AGE_MEAN=ma, DRV_AGE_35_50=((ma >= 35) & (ma <= 50)).astype(float)); p = np.clip(m7.predict(Lv), 1e-9, 1 - 1e-9)
    rows[k] = {'policies in 35–50 band': int(Lv.DRV_AGE_35_50.to_numpy()[i].sum()), 'log-lik (affected)': np.sum(y[i] * np.log(p[i]) + (1 - y[i]) * np.log(1 - p[i]))}
print(f'affected labeled policies: {i.sum()}, converted: {int(y[i].sum())}'); pd.DataFrame(rows).T.round(2)

driver age > 100: 46 rows on 46 policies (max 135); safety < 0: 79 rows (min -28.0); accident-free years > age − 16: 36% of drivers
quoted price quantiles 50/99/max: [  3760.  36569. 108608.]
affected labeled policies: 39, converted: 3


,policies in 35–50 band,log-lik (affected)
cap at 100 (current),1.0,-9.44
raw,2.0,-9.47
drop (missing),11.0,-9.84
age − 100,2.0,-9.59


**Reading:**
- **Inconclusive but harmless.** Only 39 labeled policies are affected (3 converted), and the treatments differ by ≤ 0.4 log-likelihood. The current cap fits best.
- Safety rating isn't in the formula, so its floor at 0 doesn't matter.
- **Other synthetic traces:**
  - accident-free years exceed driving years for 36% of drivers (generated independently of age);
  - price has a heavy tail (99th percentile ~$37k, max $109k).

## 6. Missing values: mechanism and imputation
SPEC7 + generator density. Residual z on rows with each kind of missing value, per era.

In [10]:
S8a = G.swap(FS.SPEC7, 'ACS_POP_DENSITY', 'DEN_KM2'); _, p = G.fit_predict(S8a, L, y)
dmiss = pd.read_parquet(I.DRV, columns=['POLICY_ID', 'HIGH_EDU_IND']); vmiss = pd.read_parquet(I.VEH, columns=['POLICY_ID', 'VEHICLE_AGE'])
M = {'ZIP': L.ZIP_CD.eq('Missing'), 'price': L.QUOTED_AMT.isna(), 'credit': L.CREDIT_SCORE.isna(), 'CAT zone': L.CAT_ZONE.isna(),
     'package': L.COV_PKG_TYPE.astype(str).eq('Missing'), 'prior carrier': L.PRIOR_CARRIER_GRP.astype(str).eq('Missing'),
     'a driver education': L.POLICY_ID.isin(dmiss[dmiss.HIGH_EDU_IND.isna()].POLICY_ID), 'a vehicle age': L.POLICY_ID.isin(vmiss[vmiss.VEHICLE_AGE.isna()].POLICY_ID)}
rows = []
for k, mk in M.items():
    mk = mk.to_numpy()
    for e in ['all', '2015', '2016', '2017-18']:
        i = mk if e == 'all' else mk & (era == e)
        rows.append({'missing': k, 'era': e, 'n': i.sum(), 'observed': y[i].mean(), 'predicted': p[i].mean(), 'z': (y[i] - p[i]).sum() / np.sqrt((p[i] * (1 - p[i])).sum())})
pd.DataFrame(rows).pivot(index='missing', columns='era', values='z').round(2)

era,2015,2016,2017-18,all
missing,,,,
CAT zone,-0.86,1.38,-1.02,-0.30
ZIP,1.10,0.46,-0.31,0.66
a driver education,-0.03,-0.42,-0.40,-0.50
a vehicle age,-2.10,0.65,-0.24,-0.90
credit,0.27,1.18,-0.27,0.78
package,-0.01,-1.14,0.06,-0.66
price,-0.18,-1.83,-0.61,-1.63
prior carrier,-0.46,0.57,-0.01,0.02


In [11]:
Lfull = I.add(L, FM.prep(pd.concat([lab, hold]))); FS.L = Lfull
VAR = {'density: missing ZIP → state mean': G.swap(S8a, 'DEN_KM2', 'DEN_KM2_IMP'),
       'CAT zone: missing → state mean': G.swap(S8a, 'CAT_ZONE', 'CAT_ZONE_IMP'),
       'package: missing → expected ordinal': G.swap(S8a, 'PKG_ORDINAL', 'PKG_ORDINAL_IMP'),
       'credit ≥ 650: missing → probability': [x for x in S8a if x['col'] != 'CREDIT_SCORE'] + [t('CREDIT_GE650_P', 'flag', '2015')],
       'education majority: missing drivers → probability': G.swap(S8a, 'R_EDU_MAJ', 'R_EDU_MAJ_P'),
       'newest vehicle ≥ 5: missing → probability': [x for x in S8a if x['col'] != 'VEH_AGE_MIN'] + [t('VEH_MIN_GE5_P', 'flag')],
       'price: missing → model imputation (R² 0.74)': G.swap(S8a, 'QUOTED_AMT', 'QUOTED_AMT_IMP')}
rows = {}
for s in [(0,), (1, 2)]:
    b = FS.run4(S8a, seeds=s)
    for k, v in VAR.items():
        r = FS.cmp(FS.run4(v, seeds=s, name='fx_priceimp_noleak' if k.startswith('price') else 'fx'), b); rows.setdefault(k, {}).update({f'Δauc {s}': r['Δauc'], f'Δll {s}': r['Δll(+better)'], f'se_ll {s}': r['se_ll']})
FS.L = L
pd.DataFrame(rows).T.style.format('{:+.5f}')

,"Δauc (0,)","Δll (0,)","se_ll (0,)","Δauc (1, 2)","Δll (1, 2)","se_ll (1, 2)"
density: missing ZIP → state mean,-0.00006,-0.35519,+0.11286,-0.00008,-0.40894,+0.27715
CAT zone: missing → state mean,+0.00001,+0.00317,+0.02904,+0.00001,+0.00062,+0.02374
package: missing → expected ordinal,+0.00001,+0.06036,+0.02133,+0.00001,+0.03705,+0.00497
credit ≥ 650: missing → probability,-0.00001,+0.00080,+0.06649,-0.00001,+0.03499,+0.08590
education majority: missing drivers → probability,+0.00022,+0.94413,+0.31045,+0.00018,+0.87676,+0.32778
newest vehicle ≥ 5: missing → probability,+0.00002,+0.08178,+0.06975,+0.00001,+0.07800,+0.05607
price: missing → model imputation (R² 0.74),+0.00003,+0.11231,+0.05177,+0.00003,+0.08896,+0.06682


**Reading:**
- **Mechanism:** no missing-value group is unusual (all |z| ≤ 2.1 over 32 checks). The values look masked after the outcome was generated.
- **Adopted** (positive on both seed sets):
  - **education majority as a probability** over drivers with missing education: +0.0002 AUC, log-likelihood z ≈ 2.7. The generator used every driver's true education;
  - package missing → expected ordinal;
  - newest vehicle ≥ 5 as a probability.
- **Not adopted:**
  - model-imputed price (out-of-fold R² 0.74 once price-derived columns are excluded) is neutral (+0.00003);
  - state-mean density for missing ZIPs hurts, so median filling is better there.

## 7. SPEC8, the remaining regional effect, blend and submissions
- **`SPEC8`** = SPEC7 with the generator density (`DEN_KM2`) and the adopted imputations (`formula_steps.SPEC8`, columns built in `formula.prep` via `impute.core`).
- **Regional adjustment:** the 2017-18 regional effect survives in SPEC8. So, inside each training fold, a shrunken (τ² = 0.03) per-ZIP3 log-odds adjustment for 2017-18 quotes is estimated from the formula's residuals and added for scoring.

In [12]:
group_table(FS.SPEC8).query("era == '2017-18'").style.format(precision=3).hide(axis='index')

level,era,real stat,sim mean,sim sd,p (sims ≥ real),sd of group effect
ZIP,2017-18,-0.978,-0.192,0.948,0.770,0.000
ZIP3,2017-18,2.545,-0.271,0.817,0.000,0.174
county,2017-18,1.893,-0.311,0.793,0.010,0.147


In [13]:
TAU2 = 0.03
late = lambda D: (D.ERA.astype(str) == '2017-18').to_numpy(); zz3 = lambda D: np.array([s[:3] for s in D.ZIP_CD.astype(str)])
def fit_z3(Ltr, ytr, tau2=TAU2):
    m = FM.Formula(FS.SPEC8, 0.1, '2era').fit(Ltr, ytr); p = m.predict(Ltr); lt = late(Ltr)
    r = pd.DataFrame({'g': zz3(Ltr)[lt], 'r': (ytr - p)[lt], 'w': (p * (1 - p))[lt]}).groupby('g').sum()
    return m, r.r / (r.w + 1 / tau2)
def pred_z3(m, eff, D):
    return 1 / (1 + np.exp(-(m.decision(D) + np.where(late(D), pd.Series(zz3(D)).map(eff).fillna(0).to_numpy(), 0))))
FOLDS = C.folds(lab); P8z = np.zeros((3, len(y)))
for s, k, tr, va in FOLDS:
    m, eff = fit_z3(L.iloc[tr], y[tr]); P8z[s, va] = pred_z3(m, eff, L.iloc[va])
PC = json.load(open('params_cats.json'))
cb = C.cv_score('cats15___copies__ohm_10', FE.add_cats(FE.add_static(lab)), PC['features'], 'cb', PC['params'])
r6, r7, r8 = (FS.run4(sp, seeds=(0, 1, 2)) for sp in (FS.SPEC6, FS.SPEC7, FS.SPEC8))
aucs = lambda Q: np.array([roc_auc_score(y[va], Q[s, va]) for s, k, tr, va in FOLDS])
v5, v6 = 0.85 * r6['oof_all'] + 0.15 * cb['oof_all'], 0.85 * r7['oof_all'] + 0.15 * cb['oof_all']
W = np.round(np.arange(0, 1.0001, 0.05), 2); wf = float(W[int(np.argmax([aucs(w * P8z + (1 - w) * cb['oof_all'])[:5].mean() for w in W]))])
cands = {'SPEC7': r7['oof_all'], 'blend_v5': v5, 'blend_v6': v6, 'SPEC8': r8['oof_all'], 'SPEC8 + ZIP3 late adjustment': P8z, f'blend_v7 = {wf} (SPEC8 + ZIP3) + {1 - wf:.1f} CatBoost': wf * P8z + (1 - wf) * cb['oof_all']}
rows = {}
for n, Q in cands.items():
    a = aucs(Q); rows[n] = {'AUC (15)': a.mean()}
    for bn, B in (('v5', v5), ('v6', v6)):
        d = a - aucs(B); rows[n][f'Δ vs blend_{bn} (1-2)'] = d[5:].mean(); rows[n][f'se {bn}'] = d[5:].std() / np.sqrt(10)
    rows[n]['top-decile'] = np.mean([C.top_decile_precision(y, Q[s]) for s in range(3)])
pd.DataFrame(rows).T.style.format({'AUC (15)': '{:.4f}', 'top-decile': '{:.1%}'} | {c: '{:+.5f}' for c in ['Δ vs blend_v5 (1-2)', 'se v5', 'Δ vs blend_v6 (1-2)', 'se v6']})

,AUC (15),Δ vs blend_v5 (1-2),se v5,Δ vs blend_v6 (1-2),se v6,top-decile
SPEC7,0.7243,+0.00030,+0.00049,-0.00035,+0.00020,32.0%
blend_v5,0.7239,+0.00000,+0.00000,-0.00065,+0.00037,31.6%
blend_v6,0.7246,+0.00065,+0.00037,+0.00000,+0.00000,31.9%
SPEC8,0.7246,+0.00064,+0.00063,-0.00001,+0.00029,32.1%
SPEC8 + ZIP3 late adjustment,0.7251,+0.00108,+0.00066,+0.00043,+0.00035,31.9%
blend_v7 = 0.9 (SPEC8 + ZIP3) + 0.1 CatBoost,0.7253,+0.00130,+0.00059,+0.00065,+0.00029,32.1%


In [14]:
H = FM.prep(hold); m, eff = fit_z3(L, y); ph = pred_z3(m, eff, H)
cbp = pd.read_csv(sorted(glob.glob('../submissions/catboost_cats5_*.csv'))[-1]); assert (cbp.POLICY_ID.values == hold.POLICY_ID.values).all()
out = {'formula_v5': C.write_submission('formula_v5', hold, ph), 'blend_v7': C.write_submission('blend_v7', hold, wf * ph + (1 - wf) * cbp.CONVERT_IND_PRED.to_numpy())}
json.dump({'spec': FS.SPEC8, 'C': 0.1, 'intercepts': '2era', 'zip3_late_offset_tau2': TAU2, 'blend_weight_formula': wf, 'zip3_effects': eff.round(4).to_dict(),
           'coefficients': m.coef_table()['coef'].round(6).to_dict(), 'submissions': {k: v.name for k, v in out.items()}}, open('formula_v5_spec.json', 'w'), indent=1, default=str)
print({k: (v.name, pd.read_csv(v).CONVERT_IND_PRED.nunique()) for k, v in out.items()})
m.coef_table()[['coef']].style.format('{:+.6f}')

{'formula_v5': ('formula_v5_20261031.csv', 7375), 'blend_v7': ('blend_v7_20261031.csv', 7375)}


,coef
term,
era=2017-18,-1.684351
CREDIT_SCORE:step[650]>=650 @2015,+0.378231
QUOTED_AMT:linear @2015,-0.000052
QUOTED_AMT:linear @2016,-0.000060
QUOTED_AMT:linear @2017-18,-0.000018
DEN_KM2:linear @2015,+0.000019
DEN_KM2:linear @2016,+0.000034
DEN_KM2:linear @2017-18,+0.000106
R_EDU_MAJ_P:flag @2017-18,+1.211971


## 8. What is the 2017-18 area effect?
1. **Screen:** 362 candidates, each as a 2017-18 term scored against SPEC8 with a pure-chance threshold from 100 simulations (`area_screen.py`). Candidates:
   - 40 area variables (2010 Census SF1 shares and densities, ACS 2011 and ACS 2018 features), each at ZIP / ZIP3-average / county-average level, raw and logged;
   - timing terms.
2. **Grouping:** which area unit carries it (in-fold shrunken offsets).
3. **Shape:** smooth (normal) vs tiers, via a nonparametric maximum-likelihood fit with a simulation null.

In [15]:
import area_screen as AS
names, Xc = AS.build(L); chi, _ = AS.score_chi2(FS.SPEC8, L, y, Xc)
m8 = FM.Formula(FS.SPEC8, 0.1, '2era').fit(L, y); p8 = m8.predict(L); rng = np.random.default_rng(17)
null = [AS.score_chi2(FS.SPEC8, L.assign(CONVERT_IND=ys), ys, Xc)[0].max() for ys in ((rng.random(len(y)) < p8).astype(float) for _ in range(100))]
print(f'{len(names)} candidates; pure-chance threshold {np.quantile(null, 0.95):.1f}')
pd.Series(chi, index=names, name='chi2').sort_values(ascending=False).head(12).round(1).to_frame()

362 candidates; pure-chance threshold 12.0


,chi2
second half 2018,6.7
days since 2017-01-01,6.6
quarter 2017Q2,5.3
quarter 2018Q3,4.0
log sf1_urbanized_share [county],3.9
sf1_urbanized_share [county],3.6
year 2018,3.6
log sf1_urban_share [county],3.5
month of year,3.5
log acs18_population [zip],3.4


In [16]:
late8 = late(L); keys = {'ZIP2': L.ZIP_CD.astype(str).str[:2].to_numpy(), 'ZIP3': zz3(L), 'county': cnty, 'ZIP': L.ZIP_CD.astype(str).to_numpy(), 'state': L.STATE_CD.astype(str).to_numpy()}
base = []
for s, k, tr, va in FOLDS:
    mm = FM.Formula(FS.SPEC8, 0.1, '2era').fit(L.iloc[tr], y[tr]); base.append((mm.predict(L.iloc[tr]), mm.decision(L.iloc[va])))
rows = {}
for gname, key in keys.items():
    for tau2 in (0.01, 0.03, 0.06):
        da = []
        for (s, k, tr, va), (ptr, zva) in zip(FOLDS, base):
            tt = late8[tr]; r = pd.DataFrame({'g': key[tr][tt], 'r': (y[tr] - ptr)[tt], 'w': (ptr * (1 - ptr))[tt]}).groupby('g').sum(); eff_ = r.r / (r.w + 1 / tau2)
            off = np.where(late8[va], pd.Series(key[va]).map(eff_).fillna(0).to_numpy(), 0)
            da.append(roc_auc_score(y[va], 1 / (1 + np.exp(-(zva + off)))) - roc_auc_score(y[va], 1 / (1 + np.exp(-zva))))
        da = np.array(da); rows[(gname, tau2)] = {'groups': len(np.unique(key[late8])), 'ΔAUC seeds 1-2': da[5:].mean(), 'se': da[5:].std() / np.sqrt(10)}
pd.DataFrame(rows).T.style.format({'groups': '{:.0f}', 'ΔAUC seeds 1-2': '{:+.5f}', 'se': '{:.5f}'})

In [17]:
GRID = np.round(np.arange(-1.2, 1.2001, 0.05), 3); i0 = list(GRID).index(0.0)
def group_ll(zlin, yy, g):
    codes, inv = np.unique(g, return_inverse=True)
    return np.column_stack([np.bincount(inv, weights=yy * (zlin + b) - np.logaddexp(0, zlin + b), minlength=len(codes)) for b in GRID])
def mix_ll(LL, w):
    M = LL.max(1, keepdims=True); return float((np.log((np.exp(LL - M) * w).sum(1)) + M[:, 0]).sum())
def npmle(LL, iters=3000):
    w = np.ones(len(GRID)) / len(GRID); E_ = np.exp(LL - LL.max(1, keepdims=True))
    for _ in range(iters):
        P = E_ * w; P /= P.sum(1, keepdims=True); w = P.mean(0)
    return w
def shape(yy):
    mm = FM.Formula(FS.SPEC8, 0.1, '2era').fit(L.assign(CONVERT_IND=yy), yy); LL = group_ll(mm.decision(L)[late8], yy[late8], zz3(L)[late8])
    taus = np.arange(0.02, 0.6, 0.02); lg = [mix_ll(LL, np.exp(-0.5 * (GRID / t_) ** 2) / np.exp(-0.5 * (GRID / t_) ** 2).sum()) for t_ in taus]
    w = npmle(LL); return {'none': LL[:, i0].sum(), 'gauss': max(lg), 'tau': taus[int(np.argmax(lg))], 'npmle': mix_ll(LL, w), 'w': w}
r = shape(y); codes, inv = np.unique(zz3(L), return_inverse=True); rng = np.random.default_rng(8); zfull = m8.decision(L); sim = []
for i in range(30):
    b = rng.normal(0, r['tau'], len(codes)); ys = (rng.random(len(y)) < 1 / (1 + np.exp(-(zfull + np.where(late8, b[inv], 0))))).astype(float); sim.append(shape(ys))
print(f"real: normal effect vs none +{r['gauss'] - r['none']:.2f} log-lik (sd {r['tau']:.2f}); nonparametric vs normal +{r['npmle'] - r['gauss']:.2f}")
print(f"simulated with a normal effect: nonparametric vs normal mean +{np.mean([s['npmle'] - s['gauss'] for s in sim]):.2f}, 95th pct +{np.quantile([s['npmle'] - s['gauss'] for s in sim], 0.95):.2f}")
print('nonparametric support (mass > 2%):', {float(GRID[i]): round(float(r['w'][i]), 3) for i in np.where(r['w'] > 0.02)[0]})

real: normal effect vs none +1.98 log-lik (sd 0.16); nonparametric vs normal +0.64
simulated with a normal effect: nonparametric vs normal mean +0.79, 95th pct +1.60
nonparametric support (mass > 2%): {-0.15: 0.028, -0.1: 0.665, -0.05: 0.045, 0.25: 0.241}


**Reading:**
- **No public variable explains the effect.** The best of 362 area and timing candidates scores χ² 6.7, against a pure-chance threshold of 12.0. That covers 2010 Census, ACS 2011 and ACS 2018 variables at ZIP, ZIP3 and county level, plus quote timing. Adding time terms also leaves the ZIP3 statistic unchanged.
- **The unit is exactly the 3-digit ZIP.** It's the only grouping whose in-fold adjustment helps (+0.00044 ± 0.00016). County helps less; ZIP, ZIP2 and state hurt. ZIP3 is a standard insurance territory unit.
- **Shape:** consistent with a normal per-ZIP3 factor (sd ≈ 0.16) applied only in 2017-18. The nonparametric fit hints at two tiers (≈ 75% at −0.1, 25% at +0.25) but gains less than chance does under a normal effect.
- **The confident high ZIP3s** are a few large ones (117 Suffolk, 088 central NJ, 112 Brooklyn, 076 Hackensack, 328 Orlando) with no common trait.
- **Conclusion:** most likely a random territory factor per ZIP3 drawn by the generator. It can only be estimated from the labels, which the shrunken ZIP3 adjustment in `blend_v7` already does.

## Conclusions
- **Density source identified exactly:** 2010 Census population ÷ 2018 Gazetteer land area, per km². The generator kept only ZIPs at ≥ 1,000/km² (1 exception in ~6,600 ZCTAs). Using it improves deviance by ~28.
- **The ZIP sample is all dense ZCTAs in 8 states**, with quotes spread roughly evenly (~30–40 per ZIP).
- **CAT zone is synthetic:** a fixed distribution per state (three types), with no geography inside a state. It's unrelated to real hazard history.
- **Capping** of the over-100 ages can't be checked (39 policies); the cap fits marginally best. Safety capping is irrelevant.
- **Missing values** were masked after the outcome was generated. Education majority as a probability helps; imputing price or density does not.
- **A 2017-18 regional (ZIP3) effect remains** that none of the tested sources explain (state, county density, ACS income/education/commute, latitude/longitude, hazard history). A shrunken in-fold ZIP3 adjustment captures part of it: +0.0004 AUC.
- **New best: `blend_v7` = 0.9 × (SPEC8 + ZIP3 adjustment) + 0.1 × CatBoost (07), 15-fold AUC 0.7253.** That's +0.0013 ± 0.0006 over `blend_v5` and +0.0007 ± 0.0003 over `blend_v6` on seeds 1–2.